## Milvus

In [1]:
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import TokenTextSplitter, CharacterTextSplitter, RecursiveCharacterTextSplitter
from pymilvus import Collection, MilvusException, connections, db, utility

from uuid import uuid4
import numpy as np
import pymupdf
import tiktoken
import dropbox

#### Embeddings

In [2]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### Vector Database

In [ ]:
from pymilvus import Collection, MilvusException, connections, db, utility

conn = connections.connect(host="localhost", port=19530)

# Create a new database
db_name = "test"
try:
    database = db.create_database(db_name)
    print(f"Database '{db_name}' created successfully.")
except:
    print(f"Database '{db_name}' already exists or could not be created.")

Database 'document_embeddings' created successfully.


In [4]:
URI = 'http://localhost:19530'

vector_store = Milvus(
    embedding_function=embeddings,
    connection_args={"uri": URI, 'token': 'root:Milvus', 'db_name': 'test'},
    index_params={"index_type": "FLAT", "metric_type": "L2"},
    consistency_level="Strong",
    drop_old=False
)

#### Filling the database

In [5]:
tokenizer = tiktoken.get_encoding("cl100k_base")

In [6]:
PDF = 'ns-en-1995-1-1_2004+a2_2014+na_2024_en_001.pdf'

In [7]:
chunk_size = 1600
chunk_overlap = 400

token_buffer = []
page_buffer = []

In [8]:
# Load pdf
doc = pymupdf.open(PDF)

rec: pymupdf.Rect = pymupdf.Rect(42, 72, 563, 772)

In [9]:
for page in doc:
    text = page.get_text('text', clip=rec)
    tokens = tokenizer.encode(text)
    token_buffer.extend(tokens)
    page_buffer.extend([page.number] * len(tokens))

    start = 0
    while len(token_buffer) - start >= chunk_size:
        end = start + chunk_size
        chunk_tokens = token_buffer[start:end]
        chunk_pages = page_buffer[start:end]
        chunk_text = tokenizer.decode(chunk_tokens)
        start += chunk_size - chunk_overlap

        metadata = {
            'Document': doc.name,
            'page_start': chunk_pages[0],
            'page_end': chunk_pages[-1],
        }
        vector_store.add_documents(
            documents=[Document(page_content=chunk_text, metadata=metadata)],
            ids=[str(uuid4())]
        )

    if start > 0:
        token_buffer = token_buffer[start:]
        page_buffer = page_buffer[start:]

In [10]:
retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'fetch_k': 100, 'lambda_mult': 0.5}
)

In [11]:
results = retriever.invoke(
    'values of kmod',
    k=10
)

len(results)

10